# 2. Preprocessing & Cleaning

Este notebook se enfoca en preparar los datos para entrenar nuestros modelos de ML y Deep Learning.

**Pasos:**
1. Cargar una muestra balanceada (ej. 100,000 para entrenamiento y 20,000 para validación/testeo).
2. Aplicar limpieza de texto avanzada.
3. Crear el pipeline de vectorización **TF-IDF** para los modelos Gradient Boosting (XGBoost/LightGBM) y guardarlo en `models/`.
4. Crear la tokenización secuencial e indexado de vocabulario para el modelo **LSTM (TensorFlow/Keras)** y guardar el diccionario `vocab.pkl`.
5. Guardar los datos procesados en la carpeta `data/processed/`.

In [ ]:
import os, sys, subprocess
from getpass import getpass
from urllib.parse import quote

REPO_URL, REPO_DIR = "https://github.com/williamsnunezp/review-nlp-pipeline.git", "/content/review-nlp-pipeline"
token = os.getenv("GITHUB_TOKEN") or getpass("GitHub token (vacío si es público): ").strip()
url = REPO_URL.replace("https://", f"https://{quote(token, safe='')}@") if token else REPO_URL

if os.path.exists(REPO_DIR):
    print("No se clonó el repo: ya existe", REPO_DIR)
else:
    result = subprocess.run(["git", "clone", url, REPO_DIR])
    if result.returncode:
        print("No se clonó el repo.")
    else:
        print(os.listdir(REPO_DIR))
        print("Preparando entorno...")
        os.makedirs("/etc/OpenCL/vendors", exist_ok=True)
        with open("/etc/OpenCL/vendors/nvidia.icd", "w") as f:
            f.write("libnvidia-opencl.so.1")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "optuna", "optuna_integration"])
        print("Entorno preparado.")


In [2]:
import os
import sys
import bz2
import pickle
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter

# Configuración estricta de rutas para Google Colab
PROJECT_ROOT = '/content/review-nlp-pipeline'
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

## 1. Carga de Muestras de Entrenamiento y Prueba
Cargaremos 120,000 registros para train/validation y 24,000 registros para test.

In [3]:
from src.preprocessing import clean_text

train_raw_path = os.path.join(PROJECT_ROOT, 'data/raw/train.csv')
test_raw_path = os.path.join(PROJECT_ROOT, 'data/raw/test.csv')

df_train_raw = pd.read_csv(train_raw_path)
df_test_raw = pd.read_csv(test_raw_path)
print(f"Datos cargados. Train: {df_train_raw.shape[0]}, Test: {df_test_raw.shape[0]}")


Datos cargados. Train: 120000, Test: 24000


## 2. Limpieza de Texto
Aplicaremos la función de limpieza y crearemos la división de validación.

In [4]:
print("Limpiando textos de entrenamiento...")
df_train_raw['cleaned_boosting'] = df_train_raw['text'].apply(lambda x: clean_text(x, model_rn=False))
df_train_raw['cleaned_rn'] = df_train_raw['text'].apply(lambda x: clean_text(x, model_rn=True))

print("Limpiando textos de prueba...")
df_test_raw['cleaned_boosting'] = df_test_raw['text'].apply(lambda x: clean_text(x, model_rn=False))
df_test_raw['cleaned_rn'] = df_test_raw['text'].apply(lambda x: clean_text(x, model_rn=True))

# Eliminar registros vacíos después de la limpieza si los hubiera
df_train_raw = df_train_raw[(df_train_raw['cleaned_boosting'] != '') & (df_train_raw['cleaned_rn'] != '')].reset_index(drop=True)
df_test_raw = df_test_raw[(df_test_raw['cleaned_boosting'] != '') & (df_test_raw['cleaned_rn'] != '')].reset_index(drop=True)

# Split de validación (15% del set de entrenamiento)
df_train, df_val = train_test_split(df_train_raw, test_size=0.15, random_state=42, stratify=df_train_raw['label'])
df_test = df_test_raw

print(f"Splits finales -> Train: {df_train.shape[0]}, Val: {df_val.shape[0]}, Test: {df_test.shape[0]}")

Limpiando textos de entrenamiento...
Limpiando textos de prueba...
Splits finales -> Train: 102000, Val: 18000, Test: 24000


## 3. Vectorización TF-IDF (para Boosting)
Entrenamos el `TfidfVectorizer` utilizando sólo el conjunto de entrenamiento para evitar el filtrado de información (*data leakage*). Las stopwords ya se eliminaron en `clean_text`.

In [5]:
max_features = 5000
vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=(1, 2))

print("Ajustando Vectorizador TF-IDF...")
X_train_tfidf = vectorizer.fit_transform(df_train['cleaned_boosting'])
X_val_tfidf = vectorizer.transform(df_val['cleaned_boosting'])
X_test_tfidf = vectorizer.transform(df_test['cleaned_boosting'])

# Guardar el vectorizador entrenado
models_dir = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(models_dir, exist_ok=True)
vectorizer_path = os.path.join(models_dir, 'vectorizer.pkl')
with open(vectorizer_path, 'wb') as f:
    pickle.dump(vectorizer, f)
print(f"Vectorizador guardado en: {vectorizer_path}")

Ajustando Vectorizador TF-IDF...
Vectorizador guardado en: /content/review-nlp-pipeline/models/vectorizer.pkl


## 4. Tokenización y Vocabulario (para LSTM y CNN)
Para los modelos, necesitamos asignar a cada palabra un ID numérico, definir el padding (rellenar/recortar secuencias a un tamaño fijo) y guardar el mapa del vocabulario.

In [10]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers
from transformers import PreTrainedTokenizerFast

MAX_VOCAB_SIZE, MAX_LEN = 11000, 100
PAD_TOKEN, UNK_TOKEN = "<PAD>", "<UNK>"

backend = Tokenizer(models.WordLevel(unk_token=UNK_TOKEN))
backend.pre_tokenizer = pre_tokenizers.WhitespaceSplit()
backend.train_from_iterator(
    df_train["cleaned_rn"].dropna().astype(str),
    trainers.WordLevelTrainer(
        vocab_size=MAX_VOCAB_SIZE, min_frequency=2,
        special_tokens=[PAD_TOKEN, UNK_TOKEN]
    )
)
tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=backend, pad_token=PAD_TOKEN, unk_token=UNK_TOKEN
)
word_to_idx = tokenizer.get_vocab()
PAD_IDX, UNK_IDX = tokenizer.pad_token_id, tokenizer.unk_token_id
vocab_path = os.path.join(models_dir, 'vocab.pkl')
with open(os.path.join(models_dir, "vocab.pkl"), "wb") as f:
    pickle.dump(word_to_idx, f)
print(f"Vocabulario guardado en: {vocab_path}")


Vocabulario guardado en: /content/review-nlp-pipeline/models/vocab.pkl


## 5. Serialización de los Datos Procesados
Guardaremos los DataFrames limpios en formato CSV para un acceso directo en la fase de modelado.

In [11]:
processed_dir = os.path.join(PROJECT_ROOT, 'data/processed')
os.makedirs(processed_dir, exist_ok=True)

cols = ['label', 'cleaned_boosting', 'cleaned_rn']
df_train[cols].to_csv(os.path.join(processed_dir, 'train.csv'), index=False)
df_val[cols].to_csv(os.path.join(processed_dir, 'val.csv'), index=False)
df_test[cols].to_csv(os.path.join(processed_dir, 'test.csv'), index=False)

print("¡Conjuntos procesados guardados con éxito en data/processed/!")

¡Conjuntos procesados guardados con éxito en data/processed/!
